# Cruces analíticos: participación y trazabilidad vs. notas (punto 6.2)

Cruza, para cada estudiante matriculado, su **participación** (tiempo promedio en la plataforma, KPI de `kpis_participacion.py`) y su **trazabilidad** (cómo usa la plataforma, KPI de `kpis_trazabilidad.py`) con su **nota** (porcentaje del total de puntos rendidos, KPI de `kpis_calificaciones.py`), y mide qué tan relacionadas están.

Contenido:
1. Tabla por alumno: asignatura, usuario, tiempo en la plataforma y nota.
2. Correlación entre tiempo y nota por asignatura.
3. Línea de tendencia y nota promedio por tramos de participación (insumos para Streamlit).
4. Gráfico de dispersión.
5. Correlación entre tiempo de conexión y calificación final, con pruebas de significancia (usa `scipy`) y tiempo según la condición académica.
6. Relación entre trazabilidad y rendimiento: indicadores de uso de la plataforma contra la nota, con y sin los estudiantes que no se presentaron.

Las funciones de este notebook reutilizan `tiempo_por_estudiante` y `nota_por_estudiante`, por lo que heredan sus criterios: solo estudiantes matriculados, tiempo acumulado en minutos y nota = suma de puntos obtenidos / suma de puntos posibles * 100. **A los estudiantes que no se presentaron a ninguna evaluación se les asigna nota 0.**

## 0. Carga de datos

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from kpis_calificaciones import nota_por_estudiante
from kpis_participacion import tiempo_por_estudiante

DATA = Path("../../data")
df_participacion = pd.read_parquet(DATA / "integrado" / "participacion_integrado.parquet")
df_calificaciones = pd.read_parquet(DATA / "integrado" / "calificaciones_integrado.parquet")
df_cruce = pd.read_parquet(DATA / "integrado" / "cruce_matriculados_integrado.parquet")

CLAVES = ["materia", "usuario"]

## 1. Tabla por alumno: participación y nota

Una fila por estudiante matriculado y asignatura. El universo son los matriculados (igual que en los KPI de origen):
- Si el estudiante no figura en el reporte de participación, su `tiempo_min` es 0.
- Si no se presentó a ningún instrumento, su `nota_pct` es **0** y `se_presento` es `False`. La columna `se_presento` permite separar a quienes sacaron 0 de quienes directamente no rindieron.

Incluir a los que no se presentaron cambia la pregunta que responde el cruce: ya no es solo si, entre quienes rindieron, más tiempo se asocia a mejor nota, sino si participar más se asocia a presentarse y obtener resultados.

In [ ]:
def participacion_vs_notas(participacion: pd.DataFrame, calificaciones: pd.DataFrame,
                           cruce: pd.DataFrame) -> pd.DataFrame:
    """Participación y nota de cada estudiante matriculado, por asignatura.

    A quienes no se presentaron a ningún instrumento se les asigna nota 0.

    Devuelve: materia, usuario, tiempo_min (tiempo acumulado en la plataforma, en minutos),
    nota_pct (0-100) y se_presento (False si no rindió ningún instrumento).
    """
    tiempo = tiempo_por_estudiante(participacion, cruce)
    notas = nota_por_estudiante(calificaciones)[CLAVES + ["nota_pct"]]
    res = tiempo.merge(notas, on=CLAVES, how="left")
    res["se_presento"] = res["nota_pct"].notna()
    res["nota_pct"] = res["nota_pct"].fillna(0)
    return res.sort_values(CLAVES).reset_index(drop=True)

In [ ]:
tabla = participacion_vs_notas(df_participacion, df_calificaciones, df_cruce)
tabla.head(10)

**Control:** cantidad de estudiantes por asignatura, cuántos se presentaron y cuántos no participaron nunca en la plataforma (0 minutos).

In [ ]:
tabla.groupby("materia").agg(
    matriculados=("usuario", "size"),
    se_presentaron=("se_presento", "sum"),
    no_se_presentaron=("se_presento", lambda s: int((~s).sum())),
    sin_participacion=("tiempo_min", lambda s: int((s == 0).sum())),
)

## 2. Correlación entre tiempo en la plataforma y nota

Se calcula por asignatura con todos los matriculados:
- **Pearson**: relación lineal entre minutos y nota.
- **Spearman**: relación entre los órdenes (rankings). Es más robusta porque el tiempo tiene una distribución muy asimétrica (pocos estudiantes con uso muy alto). Se calcula como Pearson sobre los rankings.
- **Pendiente**: puntos porcentuales de nota que cambian, en promedio, por cada hora adicional en la plataforma (regresión lineal simple).

Valores cercanos a 0 indican que no hay relación; cercanos a 1 o -1, relación fuerte.

In [ ]:
def correlacion_participacion_nota(tabla: pd.DataFrame) -> pd.DataFrame:
    """Correlación entre tiempo en la plataforma y nota, por asignatura.

    Se calcula con todas las filas de la tabla recibida. Para ver solo a quienes se presentaron,
    se filtra antes: `correlacion_participacion_nota(tabla[tabla["se_presento"]])`.

    Devuelve: materia, estudiantes, pearson, spearman, pendiente_pp_por_hora.
    """
    filas = []
    for materia, g in tabla.groupby("materia"):
        pendiente = np.polyfit(g["tiempo_min"], g["nota_pct"], 1)[0] * 60
        filas.append({
            "materia": materia,
            "estudiantes": len(g),
            "pearson": g["tiempo_min"].corr(g["nota_pct"]),
            "spearman": g["tiempo_min"].rank().corr(g["nota_pct"].rank()),
            "pendiente_pp_por_hora": pendiente,
        })
    return pd.DataFrame(filas).round(3)

In [ ]:
kpi_correlacion = correlacion_participacion_nota(tabla)
kpi_correlacion

### Lectura del resultado

La correlación sale moderada porque quienes no se presentaron usan la plataforma mucho menos que el resto. Para verificarlo, se compara el tiempo promedio de ambos grupos y se recalcula la correlación solo con quienes se presentaron:

In [ ]:
display(
    tabla.groupby(["materia", "se_presento"])["tiempo_min"]
    .agg(estudiantes="size", promedio_min="mean", mediana_min="median").round(1)
)
pd.concat(
    {
        "todos (no presentados = 0)": kpi_correlacion,
        "solo quienes se presentaron": correlacion_participacion_nota(tabla[tabla["se_presento"]]),
    },
    names=["criterio"],
).reset_index(level=0)

## 3. Insumos para el gráfico de Streamlit

El gráfico de dispersión con tendencia (`scatter_with_trend` en `ui/components/charts.py`) recibe los puntos (la tabla de la sección 1, filtrada por asignatura) y una recta de tendencia de dos puntos con las mismas columnas.

Además, `nota_por_tramo_de_participacion` agrupa a los estudiantes en tramos de igual tamaño según su tiempo en la plataforma (por defecto cuartiles) y calcula la nota promedio de cada tramo. Con una nube de puntos tan dispersa, esta vista permite ver la tendencia con más claridad.

In [ ]:
def linea_de_tendencia(tabla: pd.DataFrame) -> pd.DataFrame:
    """Recta de tendencia (regresión lineal) de la nota según el tiempo, por asignatura.

    Devuelve dos puntos por asignatura: materia, tiempo_min, nota_pct. Tiene las mismas columnas
    que la tabla de puntos, para graficar ambas juntas con `scatter_with_trend`. La recta se
    limita al rango válido de la nota (0 a 100): si el ajuste supera 100 antes del tiempo máximo
    observado, el segundo punto se corta donde la recta llega a 100.
    """
    filas = []
    for materia, g in tabla.groupby("materia"):
        pendiente, ordenada = np.polyfit(g["tiempo_min"], g["nota_pct"], 1)
        x0, x1 = g["tiempo_min"].min(), g["tiempo_min"].max()
        if pendiente > 0:
            x1 = min(x1, (100 - ordenada) / pendiente)
        elif pendiente < 0:
            x1 = min(x1, (0 - ordenada) / pendiente)
        for x in (x0, x1):
            filas.append({"materia": materia, "tiempo_min": x, "nota_pct": pendiente * x + ordenada})
    return pd.DataFrame(filas).round(2)


def nota_por_tramo_de_participacion(tabla: pd.DataFrame, tramos: int = 4) -> pd.DataFrame:
    """Nota promedio según el tramo de tiempo en la plataforma, por asignatura.

    Los estudiantes se dividen en `tramos` grupos de igual tamaño según su tiempo (cuartiles por
    defecto; el tramo 1 es el de menor participación).

    Devuelve: materia, tramo, tiempo_desde_min, tiempo_hasta_min, estudiantes, nota_promedio_pct.
    """
    piezas = []
    for materia, g in tabla.groupby("materia"):
        tramo = pd.qcut(g["tiempo_min"], tramos, duplicates="drop")
        res = (
            g.groupby(tramo, observed=True)
            .agg(tiempo_desde_min=("tiempo_min", "min"), tiempo_hasta_min=("tiempo_min", "max"),
                 estudiantes=("usuario", "size"), nota_promedio_pct=("nota_pct", "mean"))
            .reset_index(drop=True)
        )
        res.insert(0, "tramo", range(1, len(res) + 1))
        res.insert(0, "materia", materia)
        piezas.append(res)
    return pd.concat(piezas, ignore_index=True).round(2)

In [ ]:
kpi_tendencia = linea_de_tendencia(tabla)
kpi_tendencia

In [ ]:
kpi_tramos = nota_por_tramo_de_participacion(tabla)
kpi_tramos

## 4. Gráfico de dispersión

Un panel por asignatura: cada punto es un estudiante (los que no se presentaron, con nota 0, se destacan en otro color) y la línea punteada es la tendencia lineal. La imagen se guarda en `data/kpis/dispersion_tiempo_nota.png`.

In [ ]:
import matplotlib.pyplot as plt


def graficar_dispersion(tabla: pd.DataFrame, tendencia: pd.DataFrame, correlacion: pd.DataFrame,
                        ruta: Path | None = None):
    """Dispersión tiempo en la plataforma vs. nota, un panel por asignatura."""
    materias = tabla["materia"].unique()
    fig, ejes = plt.subplots(1, len(materias), figsize=(5 * len(materias), 4), sharey=True, squeeze=False)
    for eje, materia in zip(ejes[0], materias):
        p = tabla[tabla["materia"] == materia]
        t = tendencia[tendencia["materia"] == materia]
        r = correlacion.loc[correlacion["materia"] == materia, "pearson"].iloc[0]
        for presento, color, etiqueta in ((True, "#1F5C6B", "Se presentó"), (False, "#B5523B", "No se presentó")):
            q = p[p["se_presento"] == presento]
            eje.scatter(q["tiempo_min"] / 60, q["nota_pct"], s=18, alpha=0.6, color=color, label=etiqueta)
        eje.plot(t["tiempo_min"] / 60, t["nota_pct"], "--", color="#E0A458", linewidth=2)
        eje.set_title(f"{materia.capitalize()} (r = {r:.2f}, n = {len(p)})")
        eje.set_xlabel("Tiempo en la plataforma (horas)")
        eje.spines[["top", "right"]].set_visible(False)
    ejes[0][0].set_ylabel("Nota (% del máximo posible)")
    ejes[0][0].set_ylim(-5, 105)
    ejes[0][-1].legend(loc="lower right", frameon=False)
    fig.suptitle("Tiempo en la plataforma vs. nota", x=0.01, ha="left")
    fig.tight_layout()
    if ruta is not None:
        fig.savefig(ruta, dpi=150, bbox_inches="tight")
    return fig


fig = graficar_dispersion(tabla, kpi_tendencia, kpi_correlacion, DATA / "kpis" / "dispersion_tiempo_nota.png")
plt.show()

## 5. Correlación entre tiempo de conexión y calificación final

Este apartado formaliza la relación de la sección 2 con pruebas estadísticas y la vincula con la **condición académica final** (promocionado, regular, libre).

**Calificación final:** se usa `nota_pct`, el porcentaje del total de puntos rendidos (con 0 para quienes no se presentaron). Es la mejor aproximación a la calificación final con los datos disponibles; a medida que se carguen más instrumentos (por ejemplo el Parcial 2 y el recuperatorio de Álgebra), se acerca a la nota definitiva sin cambiar el cálculo.

**Dependencia:** esta sección usa `scipy` (`scipy.stats`) para obtener p-valores exactos e intervalos de confianza. Está listado en `requirements.txt`. Si no está instalado: `python -m pip install scipy` (o `conda install scipy`).

In [ ]:
from scipy import stats

from kpis_calificaciones import CONDICIONES, condicion_por_estudiante

### 5.1 Pruebas de correlación

Por asignatura se calculan:
- **Pearson** con su p-valor y un intervalo de confianza del 95 %.
- **Spearman** con su p-valor.

`significativa` es `True` cuando ambos p-valores son menores a 0,05. Se calcula para todos los matriculados y solo para quienes se presentaron.

In [ ]:
def prueba_de_correlacion(tabla: pd.DataFrame, solo_presentados: bool = False) -> pd.DataFrame:
    """Correlación entre tiempo de conexión y calificación final, con pruebas de significancia.

    Con `solo_presentados=True` se excluye a quienes no rindieron ningún instrumento.

    Devuelve: materia, estudiantes, pearson, pearson_ic95_inf, pearson_ic95_sup, pearson_p,
    spearman, spearman_p y significativa (ambos p-valores < 0,05).
    """
    datos = tabla[tabla["se_presento"]] if solo_presentados else tabla
    filas = []
    for materia, g in datos.groupby("materia"):
        pearson = stats.pearsonr(g["tiempo_min"], g["nota_pct"])
        ic = pearson.confidence_interval(0.95)
        spearman = stats.spearmanr(g["tiempo_min"], g["nota_pct"])
        filas.append({
            "materia": materia, "estudiantes": len(g),
            "pearson": pearson.statistic, "pearson_ic95_inf": ic.low, "pearson_ic95_sup": ic.high,
            "pearson_p": pearson.pvalue,
            "spearman": spearman.statistic, "spearman_p": spearman.pvalue,
        })
    res = pd.DataFrame(filas)
    res["significativa"] = (res["pearson_p"] < 0.05) & (res["spearman_p"] < 0.05)
    return res.round(4)

In [ ]:
prueba_todos = prueba_de_correlacion(tabla)
prueba_presentados = prueba_de_correlacion(tabla, solo_presentados=True)
pd.concat({"todos (no presentados = 0)": prueba_todos,
           "solo quienes se presentaron": prueba_presentados}, names=["criterio"]).reset_index(level=0)

### 5.2 Tiempo de conexión según la condición académica

Se compara el tiempo en la plataforma de cada grupo según su condición final (umbrales en porcentaje del total, definidos en `kpis_calificaciones.py`). Quienes no se presentaron a ningún instrumento se muestran aparte como *no se presentó*, para no mezclarlos con quienes se presentaron y quedaron libres.

In [ ]:
GRUPOS = CONDICIONES + ["no se presentó"]


def tiempo_por_condicion(tabla: pd.DataFrame, calificaciones: pd.DataFrame, cruce: pd.DataFrame) -> pd.DataFrame:
    """Agrega a la tabla por alumno su condición académica (promocionado, regular, libre o no se presentó)."""
    condicion = condicion_por_estudiante(calificaciones, cruce)[CLAVES + ["condicion"]]
    res = tabla.merge(condicion, on=CLAVES, how="left")
    res["condicion"] = res["condicion"].fillna("no se presentó")
    return res


def resumen_tiempo_por_condicion(tabla_condicion: pd.DataFrame) -> pd.DataFrame:
    """Tiempo en la plataforma por asignatura y condición.

    Devuelve: materia, condicion, estudiantes, promedio_min, mediana_min (incluye grupos vacíos).
    """
    res = (
        tabla_condicion.groupby(["materia", "condicion"])["tiempo_min"]
        .agg(estudiantes="size", promedio_min="mean", mediana_min="median")
    )
    completo = pd.MultiIndex.from_product([sorted(tabla_condicion["materia"].unique()), GRUPOS], names=["materia", "condicion"])
    res = res.reindex(completo).reset_index()
    res["estudiantes"] = res["estudiantes"].fillna(0).astype(int)
    return res.round(1)

In [ ]:
tabla_condicion = tiempo_por_condicion(tabla, df_calificaciones, df_cruce)
kpi_tiempo_por_condicion = resumen_tiempo_por_condicion(tabla_condicion)
kpi_tiempo_por_condicion

In [ ]:
def graficar_tiempo_por_condicion(tabla_condicion: pd.DataFrame, ruta: Path | None = None):
    """Diagrama de caja del tiempo en la plataforma (horas) según la condición, un panel por asignatura."""
    materias = tabla_condicion["materia"].unique()
    fig, ejes = plt.subplots(1, len(materias), figsize=(5 * len(materias), 4.2), sharey=True, squeeze=False)
    for eje, materia in zip(ejes[0], materias):
        p = tabla_condicion[tabla_condicion["materia"] == materia]
        grupos = [g for g in GRUPOS if (p["condicion"] == g).any()]
        datos = [p.loc[p["condicion"] == g, "tiempo_min"] / 60 for g in grupos]
        eje.boxplot(datos, tick_labels=[f"{g}\n(n={len(d)})" for g, d in zip(grupos, datos)], patch_artist=True,
                    boxprops={"facecolor": "#8FB8C2"}, medianprops={"color": "#B5523B"})
        eje.set_title(materia.capitalize())
        eje.spines[["top", "right"]].set_visible(False)
    ejes[0][0].set_ylabel("Tiempo en la plataforma (horas)")
    fig.suptitle("Tiempo de conexión según la condición académica", x=0.01, ha="left")
    fig.tight_layout()
    if ruta is not None:
        fig.savefig(ruta, dpi=150, bbox_inches="tight")
    return fig


fig = graficar_tiempo_por_condicion(tabla_condicion, DATA / "kpis" / "tiempo_por_condicion.png")
plt.show()

## 6. Relación entre trazabilidad y rendimiento

Hasta acá se midió cuánto tiempo pasa el estudiante en la plataforma. Esta sección mide **de qué manera la usa** (indicadores de trazabilidad) y qué comportamientos se asocian más a la nota. Usa la misma nota (`nota_pct`, con 0 para quienes no se presentaron) y los mismos criterios que las secciones anteriores.

**Indicadores usados**

| Indicador | Qué mide |
|---|---|
| `dias_activos` | En cuántos días distintos tuvo actividad |
| `ingresos` | Cuántas veces volvió (sesiones inferidas, con 30 min de inactividad) |
| `semanas_activas` | Constancia: en cuántas semanas distintas tuvo actividad |
| `interacciones_material` | Interacciones con el material de estudio (textos, archivos, descargas) |
| `interacciones_foros` | Interacciones en foros, temas, mensajería y contactos |

**Indicadores excluidos a propósito:** las interacciones de tipo *Evaluaciones* y *Calificaciones*, porque son consecuencia de presentarse a evaluar y se relacionarían con la nota de forma casi automática. Tampoco se usa el total de interacciones, que es casi redundante con el tiempo en la plataforma.

In [ ]:
from kpis_trazabilidad import cargar_trazabilidad, clasificar_interacciones, marcar_ingresos

df_trazabilidad = cargar_trazabilidad(DATA, solo_matriculados=True)  # eventos de estudiantes matriculados

INDICADORES = ["dias_activos", "ingresos", "semanas_activas", "interacciones_material", "interacciones_foros"]

### 6.1 Indicadores de trazabilidad por alumno

Una fila por estudiante matriculado y asignatura. Los matriculados sin ningún evento de trazabilidad quedan con 0 en todos los indicadores.

In [ ]:
def indicadores_de_trazabilidad(trazabilidad: pd.DataFrame, cruce: pd.DataFrame,
                                minutos_inactividad: int = 30) -> pd.DataFrame:
    """Indicadores de uso de la plataforma de cada estudiante matriculado, por asignatura.

    Devuelve: materia, usuario, dias_activos, ingresos, semanas_activas, interacciones_material,
    interacciones_foros. Quienes no tienen eventos quedan con 0.
    """
    eventos = marcar_ingresos(trazabilidad, minutos_inactividad)
    tipo = clasificar_interacciones(eventos)
    eventos = eventos.assign(
        semana=eventos["fecha"].dt.to_period("W"),
        es_material=tipo == "Material de estudio",
        es_foro=tipo == "Foros y mensajería",
    )
    indicadores = eventos.groupby(CLAVES).agg(
        dias_activos=("dia", "nunique"),
        ingresos=("es_ingreso", "sum"),
        semanas_activas=("semana", "nunique"),
        interacciones_material=("es_material", "sum"),
        interacciones_foros=("es_foro", "sum"),
    ).reset_index()

    matriculados = cruce.loc[cruce["matriculado"], CLAVES]
    res = matriculados.merge(indicadores, on=CLAVES, how="left").fillna(0)
    res[INDICADORES] = res[INDICADORES].astype(int)
    return res

In [ ]:
indicadores = indicadores_de_trazabilidad(df_trazabilidad, df_cruce)
indicadores.head(10)

### 6.2 Tabla por alumno: indicadores y nota

Se unen los indicadores con la nota. A quienes no se presentaron se les asigna 0 y la columna `se_presento` permite separarlos (mismo criterio que en la sección 1).

In [ ]:
def trazabilidad_vs_notas(trazabilidad: pd.DataFrame, calificaciones: pd.DataFrame,
                          cruce: pd.DataFrame) -> pd.DataFrame:
    """Indicadores de trazabilidad y nota de cada estudiante matriculado, por asignatura.

    Devuelve: materia, usuario, los indicadores de `indicadores_de_trazabilidad`, nota_pct (0-100,
    con 0 para quienes no se presentaron) y se_presento.
    """
    notas = nota_por_estudiante(calificaciones)[CLAVES + ["nota_pct"]]
    res = indicadores_de_trazabilidad(trazabilidad, cruce).merge(notas, on=CLAVES, how="left")
    res["se_presento"] = res["nota_pct"].notna()
    res["nota_pct"] = res["nota_pct"].fillna(0)
    return res.sort_values(CLAVES).reset_index(drop=True)

In [ ]:
tabla_traz = trazabilidad_vs_notas(df_trazabilidad, df_calificaciones, df_cruce)
tabla_traz.head(10)

In [ ]:
tabla_traz.groupby("materia")[INDICADORES + ["nota_pct"]].mean().round(2)

### 6.3 Correlación de cada indicador con la nota

Por asignatura y por indicador se calcula la **correlación de Spearman** (`scipy.stats.spearmanr`), más robusta que Pearson porque los indicadores tienen distribuciones muy asimétricas, junto con su p-valor exacto. Se calcula de dos formas:
- **Todos los matriculados** (los que no se presentaron cuentan con nota 0): responde si usar la plataforma de cierta manera se asocia a presentarse y obtener resultados.
- **Solo quienes se presentaron**: responde si, entre quienes rindieron, ese comportamiento se asocia a mejor nota.

Con unos 165 estudiantes por asignatura, las correlaciones débiles no son confiables, por lo que se marca como `significativa` la correlación con p < 0,05.

In [ ]:
def correlaciones_trazabilidad_nota(tabla: pd.DataFrame, solo_presentados: bool = False) -> pd.DataFrame:
    """Correlación de Spearman entre cada indicador de trazabilidad y la nota, por asignatura.

    Con `solo_presentados=True` se excluye a quienes no rindieron ningún instrumento.

    Devuelve (una fila por asignatura e indicador): materia, indicador, estudiantes, spearman,
    p_valor y significativa (p_valor < 0,05).
    """
    datos = tabla[tabla["se_presento"]] if solo_presentados else tabla
    filas = []
    for materia, g in datos.groupby("materia"):
        for indicador in INDICADORES:
            r = stats.spearmanr(g[indicador], g["nota_pct"])
            filas.append({"materia": materia, "indicador": indicador, "estudiantes": len(g),
                          "spearman": r.statistic, "p_valor": r.pvalue})
    res = pd.DataFrame(filas)
    res["significativa"] = res["p_valor"] < 0.05
    return res.round({"spearman": 3, "p_valor": 6})


def matriz_correlaciones(correlaciones: pd.DataFrame) -> pd.DataFrame:
    """Matriz indicador (filas) x asignatura (columnas) con la correlación, para `heatmap`."""
    return correlaciones.pivot(index="indicador", columns="materia", values="spearman").reindex(INDICADORES)

In [ ]:
corr_todos = correlaciones_trazabilidad_nota(tabla_traz)
corr_presentados = correlaciones_trazabilidad_nota(tabla_traz, solo_presentados=True)
corr_todos

In [ ]:
corr_presentados

**Comparación lado a lado** (correlación de Spearman; la `*` marca las significativas, p < 0,05):

In [ ]:
def con_marca(correlaciones: pd.DataFrame) -> pd.DataFrame:
    texto = correlaciones["spearman"].map("{:.2f}".format) + correlaciones["significativa"].map({True: " *", False: ""})
    return correlaciones.assign(valor=texto).pivot(index="indicador", columns="materia", values="valor").reindex(INDICADORES)


pd.concat({"todos (no presentados = 0)": con_marca(corr_todos),
           "solo quienes se presentaron": con_marca(corr_presentados)}, axis=1)

### 6.4 Gráficos

**Mapa de calor:** las dos versiones de la matriz de correlaciones, una al lado de la otra. Las celdas con `*` son las significativas. La imagen se guarda en `data/kpis/correlacion_trazabilidad_nota.png`.

In [ ]:
def graficar_mapa_correlaciones(versiones: dict[str, pd.DataFrame], ruta: Path | None = None):
    """Mapa de calor indicador x asignatura de las correlaciones; una versión por panel."""
    fig, ejes = plt.subplots(1, len(versiones), figsize=(6.5 * len(versiones), 4.2), sharey=True, squeeze=False)
    for eje, (titulo, corr) in zip(ejes[0], versiones.items()):
        matriz = matriz_correlaciones(corr)
        significativa = corr.pivot(index="indicador", columns="materia", values="significativa").reindex(INDICADORES)
        imagen = eje.imshow(matriz.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
        eje.set_xticks(range(len(matriz.columns)), [m.capitalize() for m in matriz.columns])
        eje.set_yticks(range(len(matriz.index)), matriz.index)
        for i in range(matriz.shape[0]):
            for j in range(matriz.shape[1]):
                marca = "*" if significativa.iloc[i, j] else ""
                eje.text(j, i, f"{matriz.iloc[i, j]:.2f}{marca}", ha="center", va="center", fontsize=10)
        eje.set_title(titulo)
    fig.colorbar(imagen, ax=ejes[0].tolist(), label="Correlación de Spearman con la nota", shrink=0.85)
    fig.suptitle("Trazabilidad vs. nota (* = p < 0,05)", x=0.01, ha="left")
    if ruta is not None:
        fig.savefig(ruta, dpi=150, bbox_inches="tight")
    return fig


fig = graficar_mapa_correlaciones(
    {"Todos (no presentados = 0)": corr_todos, "Solo quienes se presentaron": corr_presentados},
    DATA / "kpis" / "correlacion_trazabilidad_nota.png",
)
plt.show()

**Dispersión del indicador más relacionado:** se elige el indicador con mayor correlación promedio (en valor absoluto) en la versión que incluye a todos, y se grafica contra la nota. Los estudiantes que no se presentaron (nota 0) se destacan en otro color. La imagen se guarda en `data/kpis/dispersion_trazabilidad_nota.png`.

In [ ]:
def indicador_mas_relacionado(correlaciones: pd.DataFrame) -> str:
    """Indicador con mayor |correlación| promedio entre asignaturas."""
    return correlaciones.assign(abs_r=correlaciones["spearman"].abs()).groupby("indicador")["abs_r"].mean().idxmax()


def graficar_dispersion_trazabilidad(tabla: pd.DataFrame, indicador: str, ruta: Path | None = None):
    """Dispersión indicador de trazabilidad vs. nota, un panel por asignatura.

    Los indicadores son enteros y los puntos se apilan, por lo que se les agrega un pequeño
    desplazamiento horizontal aleatorio (solo para dibujar; no altera los datos).
    """
    rng = np.random.default_rng(0)
    materias = tabla["materia"].unique()
    fig, ejes = plt.subplots(1, len(materias), figsize=(5 * len(materias), 4.4), sharey=True, squeeze=False)
    for eje, materia in zip(ejes[0], materias):
        p = tabla[tabla["materia"] == materia]
        for presento, color, etiqueta in ((True, "#1F5C6B", "Se presentó"), (False, "#B5523B", "No se presentó")):
            q = p[p["se_presento"] == presento]
            x = q[indicador] + rng.uniform(-0.25, 0.25, len(q))
            eje.scatter(x, q["nota_pct"], s=18, alpha=0.6, color=color, label=etiqueta)
        eje.set_title(materia.capitalize())
        eje.set_xlabel(indicador.replace("_", " ").capitalize())
        eje.spines[["top", "right"]].set_visible(False)
    ejes[0][0].set_ylabel("Nota (% del máximo posible)")
    ejes[0][0].set_ylim(-5, 105)
    manijas, etiquetas = ejes[0][0].get_legend_handles_labels()
    fig.legend(manijas, etiquetas, loc="lower center", ncol=2, frameon=False)
    fig.suptitle(f"{indicador.replace('_', ' ').capitalize()} vs. nota", x=0.01, ha="left")
    fig.tight_layout(rect=[0, 0.07, 1, 1])
    if ruta is not None:
        fig.savefig(ruta, dpi=150, bbox_inches="tight")
    return fig


mejor_indicador = indicador_mas_relacionado(corr_todos)
print("Indicador más relacionado:", mejor_indicador)
fig = graficar_dispersion_trazabilidad(tabla_traz, mejor_indicador, DATA / "kpis" / "dispersion_trazabilidad_nota.png")
plt.show()